In [ ]:
import numpy as np
from datasets import load_dataset
import random
from tqdm import tqdm
from PIL import Image
import os

# =======================================================================
# 🌟 데이터셋 정보 및 상수 설정 🌟
# =======================================================================

# 데이터셋 ID: NATO & French Military Doctrine Dataset
DATASET_NAME = "racineai/VDR_Nato"

# 튜터님의 추천: 처음 실습에서는 무한한 데이터셋보다, 100개의 샘플만 빠르게 맛보기 하는 게 좋아요!
SAMPLE_COUNT = 100 
# (100개 샘플로 진행하며, 데이터셋의 기본 구조 파악 및 크로스-링구얼 쿼리 분석을 시뮬레이션합니다.)

print("===================================================================")
print(f"🛡️ [튜토리얼 시작] 데이터셋 로드: {DATASET_NAME}")
print(f"✨ 샘플 분석 개수: {SAMPLE_COUNT}개")
print("===================================================================\n")

# =======================================================================
# 🚀 1단계: 데이터 로드 전략 (스트리밍 vs. 일반 로드) 🚀
# =======================================================================
# 친절한 튜터 코멘트: Hugging Face는 데이터가 너무 클 때(12.93 GB!) 스트리밍(streaming=True)으로 로드하는 게 최고예요.
# 하지만 가끔 스트리밍이 안 되는 경우도 있으니, 안전장치(try-except)를 걸어보겠습니다!

dataset = None
try:
    # 시도 1: 스트리밍 모드 (가장 빠르고 메모리 효율적!)
    print("🔍 스트리밍 모드로 데이터셋을 테스트합니다...")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍 모드로 데이터셋 로드에 성공했습니다. (메모리 절약 짱짱!)")

except Exception as e:
    # 시도 2: 스트리밍 실패 시 (예: 환경 제약) -> 작은 샘플만 다운로드하여 강제로 진행
    print(f"⚠️ 스트리밍 로드 실패 감지 ({type(e).__name__}: {e})....")
    print("   안전을 위해 'test' 스플릿 중 아주 적은 샘플만 다운로드하여 진행합니다. 😊")
    try:
        dataset = load_dataset(DATASET_NAME, split='test', streaming=False)
        print("✅ 성공! 테스트 스플릿을 일반 모드로 로드했습니다.")
    except Exception as e_fallback:
        print(f"❌ 심각한 오류: 데이터셋 로드 자체에 실패했습니다. 오류: {e_fallback}")
        exit()


# =======================================================================
# 💻 2단계: 샘플링 및 데이터 추출 (Constraint Compliant Sampling) 💻
# =======================================================================
# 친절한 튜터 코멘트: 전체 데이터셋을 순회할 필요가 없어요! 딱 필요한 개수만 쏙 뽑아 쓰는 것이 AI 실습의 기본 기술이에요.
# len() 사용 금지, .take() 패턴 사용!

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    print("\n--- 샘플 추출 ---")
    print("✨ 스트리밍 모드 감지: take() 함수를 사용하여 샘플을 추출합니다.")
    # list()로 감싸서 반복 가능한 리스트 형태로 변환합니다.
    sampled_dataset = list(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)입니다.
    print("\n--- 샘플 추출 ---")
    print("✨ 일반 데이터셋 감지: take() 함수를 사용하여 샘플을 추출합니다.")
    sampled_dataset = list(dataset.take(SAMPLE_COUNT))


# =======================================================================
# 🧠 3단계: AI 실습 시뮬레이션 - 크로스-링구얼 VQA & 문서 검색 시뮬레이션 🧠
# =======================================================================
# 실습 목표: 각 샘플이 가진 이미지, 영어 질문, 프랑스어 질문을 확인하며,
# "이 질문 쌍이 이 이미지와 관련이 깊은지"를 사람이 직접 검토하는 과정(인간의 지식 활용)을 시뮬레이션합니다.

print("\n" + "="*80)
print("🧠 실습 시작: Cross-Lingual VQA & Document Retrieval 시뮬레이션")
print("   - 우리의 목표: 이미지(Visual)와 두 언어의 질문(Text)을 연결하는 로직을 만듭니다.")
print("="*80)

print(f"\n[📌 데이터 분석 참고] 데이터셋의 핵심 필드 구조는 다음과 같습니다:")
print("  - page_num: 페이지 번호 (정량적 분석에 유용!)")
print("  - query_en: 영어 질문 (English Query)")
print("  - query_fr: 프랑스어 질문 (French Query)")
print("  - image: 실제 이미지 데이터 (Visual Input)")


for i, sample in enumerate(tqdm(sampled_dataset, desc="🔍 샘플 분석 진행 중")):
    
    # 1. 정량적 정보 추출 (메타데이터)
    page_num = sample['page_num']
    total_pages = sample['total_pages']
    
    # 2. 쿼리 및 언어 정보 추출 (텍스트 데이터)
    query_en = sample['query_en']
    query_fr = sample['query_fr']
    
    # 3. 시각 데이터 추출 및 전처리 (Image Input)
    image_data = sample['image']
    
    # 이미지의 형태를 확인하여 전처리 시뮬레이션
    if isinstance(image_data, np.ndarray):
        # NumPy 배열인 경우, 차원 정보 출력
        image_shape_info = f"Shape: {image_data.shape} (NumPy Array)"
    elif isinstance(image_data, Image.Image):
        # PIL Image 객체인 경우, 크기 정보 출력
        image_shape_info = f"Size: {image_data.size} (PIL Image)"
    else:
        image_shape_info = "Unknown Image Type"


    # -------------------------------------------------------
    # 💡 핵심 실습: 데이터의 연결성 확인 및 시뮬레이션 💡
    # -------------------------------------------------------
    
    print(f"\n--- 🖼️ 샘플 #{i+1} 분석 (Page: {page_num}/{total_pages}) ---")
    print(f"  [🖼️ 시각 정보] {image_shape_info}")
    
    # 1차 시뮬레이션: 크로스-링구얼 쌍 확인
    print(f"  [🇬🇧 English Query] >> {query_en[:60]}... (질문 내용 일부)")
    print(f"  [🇫🇷 French Query] >> {query_fr[:60]}... (질문 내용 일부)")
    
    # 2차 시뮬레이션: 검색 로직 추론
    print("  [🤖 검색 로직 시뮬레이션]")
    print("  - 이처럼 한 이미지에 영어와 프랑스어 질문이 동시에 붙어있다는 것은, 이 이미지가 'NATO' 또는 '국방 교리'와 같은 **다국적, 다층위의 공식적 주제**를 다루고 있음을 의미합니다.")
    print("  - 다음 단계는 LLM (거대 언어 모델)을 사용하여 '이 이미지'와 '이 질문 쌍'의 논리적 관계 점수를 매기는 것입니다. (RAG 기반 검색 시스템의 핵심!)")


# =======================================================================
# 🎉 마무리 인사 및 요약 🎉
# =======================================================================

print("\n" + "="*80)
print("🥳 🎉 실습 성공! 축하합니다! 🎉 🥳")
print("="*80)
print("🎉 튜터 코멘트:")
print("1. 데이터 로드: 스트리밍 모드 처리, 데이터셋 핸들링 능력을 기르셨어요! (매우 중요!)")
print("2. 크로스-링구얼 능력: 영어와 프랑스어 질문이 한 이미지에 같이 붙어 있다는 구조(bilingual, multimodal)를 이해하셨습니다.")
print("3. 실질적인 AI 응용: 이 데이터를 사용하면 단순 이미지 분류를 넘어, '이 이미지에 대해 영어로 질문하면, 관련 교리서의 특정 페이지(page_num)가 검색되는' 고난도 검색 시스템을 구축할 수 있게 됩니다.")
print("\n다음 시간에는 이 데이터를 가지고 실제 '질문-이미지 검색' 모델을 구현해보면 완벽할 거예요! 🌟")